# TensorFlow Serving REST/gRPC, model versioning

The notebook demonstrates a model server for running and maintaining models. The model server allows you to conveniently manage model versions in production.

In [ ]:
! pip install tensorflow tensorflow-serving-api grpcio

   ━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/572.6 MB 11.5 MB/s eta 0:00:32

In [ ]:
from pathlib import Path
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train[:10000], y_train[:10000], validation_split=0.1, epochs=2, batch_size=128)

export_path = Path("models/mnist/1")
export_path.mkdir(parents=True, exist_ok=True)
model.export(str(export_path))
print(f"SavedModel exported to: {export_path.resolve()}")


## Launching TensorFlow Serving via Docker

In the terminal, from the project directory:
```
docker compose
```

command for tensorflow/serving
```bash
docker run -p 8501:8501 -p 8500:8500 \
  --name tfserving_mnist \
  --rm \
  -v "$PWD/models/mnist:/models/mnist" \
  -e MODEL_NAME=mnist \
  tensorflow/serving
```

Port `8501` for REST, and `8500` for gRPC.


In [ ]:
import requests

image = x_test[0].tolist()
payload = {"instances": [image]}

response = requests.post(
    "http://host.docker.internal:8501/v1/models/mnist:predict",
    json=payload,
)

print(response.status_code)
print(response.text)


## Model Versioning

To add a new version, retrain the model and save it to the directory:

```text
models/mnist/2/
```

TensorFlow Serving can handle new model versions without changing the client code.


## gRPC - szkic klienta

gRPC zwykle jest bardziej wydajne niż REST przy komunikacji wewnętrznej między usługami, ale jest mniej wygodne do ręcznego debugowania.


In [ ]:
import numpy as np
import grpc
import tensorflow as tf
from tensorflow_serving.apis import prediction_service_pb2_grpc, predict_pb2

channel = grpc.insecure_channel("host.docker.internal:8500")

stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)

request = predict_pb2.PredictRequest()
request.model_spec.name = "mnist"
request.model_spec.signature_name = "serving_default"

image = x_test[0:1].astype(np.float32)  # shape: (1, 28, 28)

request.inputs["input_1"].CopyFrom(
    tf.make_tensor_proto(image, dtype=tf.float32, shape=[1, 28, 28])
)

result = stub.Predict(request, timeout=10.0)
print(result)

In [ ]:
print(y_test[0])

## Summary

- Flask/FastAPI are great for getting started.
- TensorFlow Serving is a specialized model server.
- Key differences: model versions, REST/gRPC, canary/A/B testing, and lower overhead.
- In production, model deployment is a process, not just a single `.h5` or `.keras` file.